# Lesson 0.2: Types of Data

Build a small student table with a nominal, an ordinal, a discrete and a continuous column, and see which summaries make sense for each.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. Build a table with four kinds of data

One column per type: eye colour (nominal), grade level (ordinal, so we tell pandas the order), siblings (discrete, counted) and height (continuous, measured). The dtypes show how pandas stores each one.

In [1]:
import pandas as pd

df = pd.DataFrame({
    'eye': ['brown','blue','brown','green','brown','blue','hazel','brown','green','blue'],
    'grade': ['Sophomore','Senior','Freshman','Junior','Sophomore','Senior','Freshman','Sophomore','Junior','Senior'],
    'siblings': [1, 0, 2, 3, 1, 1, 0, 2, 4, 1],
    'height_cm': [172.4, 165.1, 180.3, 158.9, 175.6, 169.8, 162.2, 177.0, 171.3, 183.5],
})
df['eye'] = df['eye'].astype('category')
df['grade'] = pd.Categorical(df['grade'],
    categories=['Freshman','Sophomore','Junior','Senior'], ordered=True)
print(df.dtypes)

eye          category
grade        category
siblings        int64
height_cm     float64
dtype: object


## 2. Count the categories

For categories the natural summary is a count (or a percentage), not an average. value_counts does it. With sort=False the ordinal column stays in its natural order, from Freshman to Senior.

In [2]:
print(df['eye'].value_counts())
print()
print(df['grade'].value_counts(sort=False))
print()
print('share Junior or above:', (df['grade'] >= 'Junior').mean())

eye
brown    4
blue     3
green    2
hazel    1
Name: count, dtype: int64

grade
Freshman     2
Sophomore    3
Junior       2
Senior       3
Name: count, dtype: int64

share Junior or above: 0.5


## 3. Describe the numbers

Counts and measurements are real amounts, so mean, standard deviation and quartiles all make sense. describe gives them in one table.

In [3]:
print(df[['siblings', 'height_cm']].describe().round(2))

       siblings  height_cm
count     10.00      10.00
mean       1.50     171.61
std        1.27       7.88
min        0.00     158.90
25%        1.00     166.28
50%        1.00     171.85
75%        2.00     176.65
max        4.00     183.50


## 4. Why averaging a category makes no sense

Pandas refuses to average a category. If you force it by turning grades into codes 1 to 4, you get a number like 2.6 that is not a grade, and it pretends the gaps between grades are equal. For an ordinal column use counts, the maximum or a share instead.

In [4]:
for col in ['eye', 'grade']:
    try:
        df[col].mean()
    except TypeError:
        print(f'mean of {col}: not allowed (TypeError)')

codes = df['grade'].cat.codes + 1
print('mean of grade codes:', codes.mean(), '(is that a grade? no)')
print('highest grade present:', df['grade'].max())

mean of eye: not allowed (TypeError)
mean of grade: not allowed (TypeError)
mean of grade codes: 2.6 (is that a grade? no)
highest grade present: Senior


## Try this

- Add a new column 'satisfaction' with the ordered categories Poor, OK, Good and ask for its value_counts. Which summary would you report?
- Change one height to 250.0 and run describe again. Which numbers move, and which do not?

## The same idea in R, SPSS and Stata

**R**

```r
eye <- factor(c('brown','blue','brown','green','brown','blue','hazel','brown','green','blue'))
grade <- factor(c('Sophomore','Senior','Freshman','Junior','Sophomore','Senior','Freshman','Sophomore','Junior','Senior'),
  levels = c('Freshman','Sophomore','Junior','Senior'), ordered = TRUE)
siblings <- c(1, 0, 2, 3, 1, 1, 0, 2, 4, 1)
height_cm <- c(172.4, 165.1, 180.3, 158.9, 175.6, 169.8, 162.2, 177.0, 171.3, 183.5)
table(eye)
table(grade)
summary(siblings)
summary(height_cm)
mean(eye)
```

**SPSS** (Analyze > Descriptive Statistics > Frequencies (categories) and Descriptives (numbers))

```
DATA LIST FREE / eye (A6) grade siblings height.
BEGIN DATA
brown 2 1 172.4 blue 4 0 165.1 brown 1 2 180.3 green 3 3 158.9 brown 2 1 175.6
blue 4 1 169.8 hazel 1 0 162.2 brown 2 2 177.0 green 3 4 171.3 blue 4 1 183.5
END DATA.
VALUE LABELS grade 1 'Freshman' 2 'Sophomore' 3 'Junior' 4 'Senior'.
VARIABLE LEVEL eye (NOMINAL) /grade (ORDINAL) /siblings height (SCALE).
FREQUENCIES VARIABLES=eye grade.
DESCRIPTIVES VARIABLES=siblings height.
```

**Stata**

```
clear
set obs 10
generate eye = word("brown blue brown green brown blue hazel brown green blue", _n)
generate grade = real(word("2 4 1 3 2 4 1 2 3 4", _n))
generate siblings = real(word("1 0 2 3 1 1 0 2 4 1", _n))
generate height = real(word("172.4 165.1 180.3 158.9 175.6 169.8 162.2 177.0 171.3 183.5", _n))
label define g 1 "Freshman" 2 "Sophomore" 3 "Junior" 4 "Senior"
label values grade g
tabulate eye
tabulate grade
summarize siblings height
```